In [1]:
!pip install -q "transformers>=4.51.0" peft accelerate scikit-learn matplotlib pandas requests sentencepiece


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

True
NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
!pip install -q -U "transformers>=4.51.0" peft accelerate sentencepiece


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
#!/usr/bin/env python
# =============================================================================
# HDFS block-level anomaly detection with Qwen3-0.6B + LoRA  (Google Colab, GPU)
#
# Run in Colab:   !python hdfs_qwen3_lora.py        (or paste into one cell)
# Smoke test:     !TRAIN_N=400 VAL_N=100 TEST_N=200 EPOCHS=1 python hdfs_qwen3_lora.py
#
# Data: uses labeled HDFS_v1 files if present under /content (Event_traces.csv,
# anomaly_label.csv, HDFS_templates.csv, ...). If none are found it tries to
# download HDFS_v1 from Zenodo. If that fails, upload the files to /content.
# =============================================================================
import os, sys, re, gc, json, time, math, random, shutil, zipfile, inspect, subprocess, contextlib
from collections import Counter

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def _ensure_deps():
    need = []
    try:
        import transformers
        from packaging import version
        if version.parse(transformers.__version__) < version.parse("4.51.0"):
            need.append("transformers>=4.51.0")
    except Exception:
        need.append("transformers>=4.51.0")
    for mod, pkg in [("peft", "peft"), ("accelerate", "accelerate"), ("sklearn", "scikit-learn"),
                     ("matplotlib", "matplotlib"), ("pandas", "pandas"), ("requests", "requests")]:
        try:
            __import__(mod)
        except Exception:
            need.append(pkg)
    if need:
        print("Installing:", need)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])


_ensure_deps()

import numpy as np
import pandas as pd
import requests
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from transformers import (AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments,
                          DataCollatorForSeq2Seq, set_seed)
from peft import LoraConfig, get_peft_model, PeftModel

# ----------------------------------------------------------------------------- config
SEED = 42
MODEL_NAME = "Qwen/Qwen3-0.6B"
ROOT = "/content" if os.path.isdir("/content") else os.getcwd()
DATA_DIR = f"{ROOT}/hdfs_data"
ADAPTER_DIR = f"{ROOT}/qwen3-hdfs-lora"
CKPT_DIR = f"{ROOT}/qwen3-hdfs-ckpt"
PRED_CSV = f"{ROOT}/hdfs_test_predictions.csv"
METRICS_JSON = f"{ROOT}/hdfs_metrics.json"
CM_PNG = f"{ROOT}/hdfs_confusion_matrix.png"

# Split is ALWAYS done on the full trace set (80/10/10, block level). To keep Colab runtime
# reasonable, each split is then subsampled (env-overridable; TEST_N=0 -> full test split).
TRAIN_N = int(os.environ.get("TRAIN_N", 8000))
VAL_N = int(os.environ.get("VAL_N", 600))
TEST_N = int(os.environ.get("TEST_N", 6000))
# Train/val subsets are enriched with anomalies (HDFS is ~3% anomalous) so the model sees enough of
# them. The TEST subset keeps the natural class ratio. The majority baseline uses the natural
# training-split prior. Set to None for a natural-ratio training subset.
TRAIN_ANOMALY_FRAC = 0.25
EPOCHS = int(os.environ.get("EPOCHS", 2))
LR = 2e-4
WEIGHT_DECAY = 0.01
EFFECTIVE_BS = 32
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
MAX_LEN_CAP = 1024
GEN_MAX_NEW_TOKENS = 16
DEDUP_SEQUENCES = False  # True: keep one trace per identical event sequence (changes distribution)

NORMAL, ANOMALY, INVALID = "Normal", "Anomaly", "INVALID"
PROMPT = """You are an HDFS log anomaly detection system.
Analyze the following HDFS block event sequence and classify it as exactly one of:
Normal
Anomaly

Event sequence:
{seq}

Return ONLY:
Normal
or:
Anomaly"""

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); set_seed(SEED)
CUDA = torch.cuda.is_available()
DEVICE = "cuda" if CUDA else "cpu"
USE_BF16 = CUDA and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float32   # fp32 weights + fp16 AMP on T4-class GPUs
USE_FP16_AMP = CUDA and not USE_BF16
print(f"CUDA available: {CUDA}" + (f" | {torch.cuda.get_device_name(0)}" if CUDA else ""))
if not CUDA:
    print("WARNING: no GPU detected. This will be extremely slow. Runtime > Change runtime type > GPU.")


def hr(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)


def human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.1f}{u}"
        n /= 1024
    return f"{n:.1f}TB"


# ============================================================================= 1. discovery
def count_lines(path):
    n = 0
    with open(path, "rb") as f:
        while True:
            chunk = f.read(1 << 24)
            if not chunk:
                break
            n += chunk.count(b"\n")
    return n


def discover():
    roots = [r for r in dict.fromkeys([ROOT, "/content", "/kaggle/input", "/root", "/mnt/data", os.getcwd()])
             if os.path.isdir(r)]
    pat = re.compile(r"hdfs|event_traces|anomaly_label", re.I)
    skip = {".git", "site-packages", "__pycache__", "qwen3-hdfs-ckpt", "qwen3-hdfs-lora", "sample_data"}
    found, seen = [], set()
    for r in roots:
        for dp, dn, fn in os.walk(r):
            dn[:] = [d for d in dn if d not in skip and not d.startswith(".")]
            for f in fn:
                p = os.path.join(dp, f)
                rp = os.path.realpath(p)
                if pat.search(f) and rp not in seen and os.path.isfile(p):
                    seen.add(rp)
                    found.append(p)
    return sorted(found)


def describe(p):
    sz = os.path.getsize(p)
    print(f"\n{p}  [{human(sz)}]")
    low = p.lower()
    try:
        if low.endswith(".csv"):
            head = pd.read_csv(p, nrows=3)
            print(f"  rows: {count_lines(p) - 1:,}   columns: {list(head.columns)}")
            print("  " + head.to_string(max_colwidth=70).replace("\n", "\n  "))
        elif low.endswith(".npz") and sz < 300 * 2**20:
            z = np.load(p, allow_pickle=True)
            print("  npz keys/shapes:", {k: getattr(z[k], "shape", None) for k in z.files})
    except Exception as e:
        print("  (could not preview)", e)


def download_hdfs_v1():
    os.makedirs(DATA_DIR, exist_ok=True)
    zpath = os.path.join(DATA_DIR, "HDFS_v1.zip")
    urls = ["https://zenodo.org/records/8196385/files/HDFS_v1.zip?download=1",
            "https://zenodo.org/record/3227177/files/HDFS_v1.zip?download=1"]
    for u in urls:
        try:
            print("Downloading", u)
            with requests.get(u, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(zpath, "wb") as f:
                    for ch in r.iter_content(1 << 20):
                        f.write(ch)
            return zpath
        except Exception as e:
            print("  failed:", e)
    raise SystemExit("Could not download HDFS_v1. Upload Event_traces.csv + anomaly_label.csv "
                     f"(+ HDFS_templates.csv) to {ROOT} and re-run.")


def extract_zip(zpath):
    with zipfile.ZipFile(zpath) as z:
        for m in z.infolist():
            if m.filename.lower().endswith(".log"):   # skip the 1.5GB raw log; not needed
                continue
            z.extract(m, DATA_DIR)
    print("Extracted", zpath, "->", DATA_DIR)


hr("1. DATASET DISCOVERY")
files = discover()
usable = lambda fs: any(re.search(r"event_traces|anomaly_label|log_structured", os.path.basename(p), re.I)
                        for p in fs if p.lower().endswith(".csv"))
if not usable(files):
    zips = [p for p in files if p.lower().endswith(".zip")]
    extract_zip(zips[0] if zips else download_hdfs_v1())
    files = discover()
print(f"Discovered {len(files)} HDFS-related files")
for p in files:
    describe(p)

# ============================================================================= 2. structure
LABEL_MAP = {"normal": NORMAL, "success": NORMAL, "anomaly": ANOMALY, "fail": ANOMALY, "failure": ANOMALY,
             "abnormal": ANOMALY, "anomalous": ANOMALY, "0": NORMAL, "1": ANOMALY}
BLK_RE = re.compile(r"^\s*blk_-?\d+\s*$")
EV_RE = re.compile(r"\bE\d+\b")


def norm_label(v):
    return LABEL_MAP.get(str(v).strip().lower())


def infer_cols(df, name):
    obj = [c for c in df.columns if df[c].dtype == object or str(df[c].dtype).startswith("str")]
    samp = lambda c: df[c].dropna().astype(str).head(1000)
    block = next((c for c in obj if len(samp(c)) and samp(c).map(lambda s: bool(BLK_RE.match(s))).mean() >= 0.9), None)
    label = None
    for c in df.columns:
        if c == block:
            continue
        u = set(df[c].dropna().astype(str).str.strip().str.lower().unique())
        if 1 <= len(u) <= 4 and u <= set(LABEL_MAP):
            if label is None or "label" in c.lower():
                label = c
    seq, best = None, 0.0
    for c in obj:
        if c in (block, label):
            continue
        s = samp(c)
        if len(s) == 0:
            continue
        cnt = s.map(lambda x: len(EV_RE.findall(x)))
        if (cnt >= 1).mean() >= 0.9 and cnt.mean() > best:
            seq, best = c, cnt.mean()
    print(f"[{name}] inferred -> block: {block!r} | label: {label!r} | event sequence: {seq!r}")
    if label is not None:
        u = sorted(df[label].dropna().astype(str).unique())
        print(f"[{name}] label values {u} -> {[norm_label(x) for x in u]}")
        if set(x.strip().lower() for x in u) <= {"0", "1"}:
            print("  WARNING: numeric labels interpreted as 1 = Anomaly")
    return block, label, seq


def pick(pattern):
    c = [p for p in files if p.lower().endswith(".csv") and re.search(pattern, os.path.basename(p), re.I)]
    c.sort(key=lambda p: ("v2" in p.lower(), -os.path.getsize(p)))
    return c[0] if c else None


def traces_from_structured(path):
    df = pd.read_csv(path)
    obj = [c for c in df.columns if df[c].dtype == object]
    content = next(c for c in obj if df[c].astype(str).str.contains(r"blk_-?\d+").mean() > 0.5)
    ev = next(c for c in obj if c != content and df[c].astype(str).str.fullmatch(r"E\d+").mean() > 0.9)
    print(f"[structured log] content col: {content!r}, event col: {ev!r}")
    if "LineId" in df.columns:
        df = df.sort_values("LineId")
    df["_blk"] = df[content].astype(str).str.findall(r"blk_-?\d+")
    ex = df[[ev, "_blk"]].explode("_blk").dropna(subset=["_blk"])
    g = ex.groupby("_blk", sort=False)[ev].apply(lambda s: " ".join(s)).reset_index()
    g.columns = ["block_id", "event_sequence"]
    return g


hr("2. SUPERVISED STRUCTURE")
p_et, p_al, p_tp, p_sl = pick(r"event_traces"), pick(r"anomaly_label"), pick(r"template"), pick(r"log_structured")
print("Using:", {"event_traces": p_et, "anomaly_label": p_al, "templates": p_tp, "structured": p_sl})

emb_label = None
if p_et:
    d = pd.read_csv(p_et)
    b, l, s = infer_cols(d, "Event_traces")
    if b is None or s is None:
        raise SystemExit("Could not infer block/sequence columns from the event-traces file.")
    traces = pd.DataFrame({"block_id": d[b].astype(str).str.strip(),
                           "event_sequence": d[s].astype(str).map(lambda x: " ".join(EV_RE.findall(x)))})
    if l is not None:
        emb_label = d[l].map(norm_label)
elif p_sl:
    traces = traces_from_structured(p_sl)
else:
    raise SystemExit("No event-traces or structured-log CSV found.")

label_src = None
if p_al:
    da = pd.read_csv(p_al)
    ba, la, _ = infer_cols(da, "anomaly_label")
    if ba is None or la is None:
        raise SystemExit("Could not infer block/label columns from the anomaly-label file.")
    lab = pd.DataFrame({"block_id": da[ba].astype(str).str.strip(), "label": da[la].map(norm_label)})
    lab = lab.drop_duplicates("block_id")
    traces = traces.merge(lab, on="block_id", how="left")
    label_src = "anomaly_label.csv"
    if emb_label is not None:
        cmp = traces.assign(_emb=emb_label.values).dropna(subset=["label", "_emb"])
        print(f"Cross-check Event_traces label vs anomaly_label.csv: {(cmp.label != cmp._emb).sum()} mismatches "
              f"out of {len(cmp):,}")
elif emb_label is not None:
    traces["label"] = emb_label.values
    label_src = "label column inside event-traces file"
else:
    raise SystemExit("No labels available (HDFS_v2 / raw logs are unlabeled). Provide anomaly_label.csv.")
print("Label source:", label_src)

templates = {}
if p_tp:
    dt = pd.read_csv(p_tp)
    ic = next((c for c in dt.columns if "id" in c.lower()), dt.columns[0])
    tc = next((c for c in dt.columns if "template" in c.lower()), dt.columns[-1])
    templates = dict(zip(dt[ic].astype(str), dt[tc].astype(str)))
    print(f"Templates loaded: {len(templates)}; e.g. {list(templates.items())[:2]}")

print("\nExample complete traces (block -> events -> label):")
for _, r in traces.dropna(subset=["label"]).groupby("label").head(2).iterrows():
    ev = r.event_sequence.split()
    print(f"Block: {r.block_id}\nEvents: {' → '.join(ev[:25])}{' → …' if len(ev) > 25 else ''}  ({len(ev)} events)\n"
          f"Label: {r.label}\n")

# ============================================================================= 3. preprocessing
hr("3. PREPROCESSING")
n0 = len(traces)
traces = traces[traces.block_id.map(lambda s: bool(BLK_RE.match(s)))]
n_malformed = n0 - len(traces)
n1 = len(traces)
traces = traces.dropna(subset=["label"])
n_nolabel = n1 - len(traces)
n2 = len(traces)
traces = traces[traces.event_sequence.str.len() > 0]
n_empty = n2 - len(traces)
n3 = len(traces)
conf = traces.groupby("block_id").label.nunique()
traces = traces[~traces.block_id.isin(conf[conf > 1].index)]
n_conflict = n3 - len(traces)
n4 = len(traces)
traces = traces.drop_duplicates("block_id")
n_dupblock = n4 - len(traces)
n_dupseq = 0
if DEDUP_SEQUENCES:
    n5 = len(traces)
    traces = traces.drop_duplicates(["event_sequence", "label"])
    n_dupseq = n5 - len(traces)
traces = traces.reset_index(drop=True)
traces["n_events"] = traces.event_sequence.str.count(" ") + 1
print(f"Removed: malformed={n_malformed} missing_label={n_nolabel} empty={n_empty} "
      f"conflicting_label_blocks={n_conflict} duplicate_block_rows={n_dupblock} duplicate_sequences={n_dupseq}")
vc = traces.label.value_counts()
print(f"Total traces: {len(traces):,}\nNormal: {vc.get(NORMAL, 0):,}\nAnomaly: {vc.get(ANOMALY, 0):,}")
print(f"Class ratio (Normal:Anomaly) = {vc.get(NORMAL, 0) / max(vc.get(ANOMALY, 1), 1):.2f}:1 | "
      f"anomaly share = {vc.get(ANOMALY, 0) / len(traces):.4%}")
print("Sequence length (events):", traces.n_events.describe(percentiles=[.5, .95, .99]).round(2).to_dict())
print("By class median length:", traces.groupby("label").n_events.median().to_dict())
print(f"Unique event sequences: {traces.event_sequence.nunique():,}")
seq_conflict = (traces.groupby("event_sequence").label.nunique() > 1).sum()
print(f"Identical sequences carrying BOTH labels: {seq_conflict:,}")
TRUTH = dict(zip(traces.block_id, traces.label))

# ============================================================================= 4. split (block level)
hr("4. BLOCK-LEVEL SPLIT (seed=42)")
strat_ok = traces.label.value_counts().min() >= 10
if not strat_ok:
    print("WARNING: a class has <10 traces -> stratification impossible; using deterministic non-stratified split.")
train_full, tmp = train_test_split(traces, test_size=0.2, random_state=SEED,
                                   stratify=traces.label if strat_ok else None)
val_full, test_full = train_test_split(tmp, test_size=0.5, random_state=SEED,
                                       stratify=tmp.label if strat_ok else None)
for a, b, n in [(train_full, val_full, "train/val"), (train_full, test_full, "train/test"), (val_full, test_full, "val/test")]:
    assert not (set(a.block_id) & set(b.block_id)), f"block leakage {n}"
print("Block-level leakage check: PASSED (no block ID in more than one split)")


def split_counts(df):
    v = df.label.value_counts()
    return int(v.get(NORMAL, 0)), int(v.get(ANOMALY, 0))


for nm, d in [("Train", train_full), ("Validation", val_full), ("Test", test_full)]:
    n, a = split_counts(d)
    print(f"{nm}: {len(d):,}\n  Normal: {n:,}\n  Anomaly: {a:,}")

train_seq_set = set(train_full.event_sequence)
overlap = test_full.event_sequence.isin(train_seq_set).mean()
print(f"Note: {overlap:.1%} of test traces have an event sequence identical to some training trace "
      "(different blocks; reported separately below as seen vs unseen).")


def subsample(df, n, frac, seed):
    if not n or n >= len(df):
        return df.sample(frac=1, random_state=seed).reset_index(drop=True)
    if frac is None:
        s, _ = train_test_split(df, train_size=n, random_state=seed, stratify=df.label)
        return s.reset_index(drop=True)
    na = min(int(round(n * frac)), int((df.label == ANOMALY).sum()))
    nn = min(n - na, int((df.label == NORMAL).sum()))
    s = pd.concat([df[df.label == ANOMALY].sample(na, random_state=seed),
                   df[df.label == NORMAL].sample(nn, random_state=seed)])
    return s.sample(frac=1, random_state=seed).reset_index(drop=True)


train_df = subsample(train_full, TRAIN_N, TRAIN_ANOMALY_FRAC, SEED)
val_df = subsample(val_full, VAL_N, TRAIN_ANOMALY_FRAC, SEED)
test_df = subsample(test_full, TEST_N, None, SEED)
print("\nSubsets actually used (compute budget):")
for nm, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    n, a = split_counts(d)
    print(f"  {nm}: {len(d):,} (Normal {n:,} / Anomaly {a:,})")

# ============================================================================= 5/6. chat format
hr("5-6. QWEN CHAT FORMAT + INSPECTION")
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"   # for generation; the training collator pads via its own logic below


def prompt_text(seq):
    msgs = [{"role": "user", "content": PROMPT.format(seq=seq)}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)


def encode(seq, label):
    p = tok(prompt_text(seq), add_special_tokens=False).input_ids
    a = tok(label + "<|im_end|>", add_special_tokens=False).input_ids
    return {"input_ids": p + a, "attention_mask": [1] * (len(p) + len(a)), "labels": [-100] * len(p) + a}


for i in range(5):
    r = train_df.iloc[i]
    pt = prompt_text(r.event_sequence)
    assert TRUTH[r.block_id] == r.label and r.event_sequence in pt and r.label in (NORMAL, ANOMALY)
    print(f"--- example {i + 1} | block {r.block_id}\nINPUT:\n{pt}\nOUTPUT:\n{r.label}\n")
assert not (set(train_df.block_id) & (set(val_df.block_id) | set(test_df.block_id)))
assert not (set(val_df.block_id) & set(test_df.block_id))
assert (train_df.event_sequence.str.len() > 0).all()
print("Checks passed: labels match source, sequences present, prompts well-formed, no train/val/test leakage.")

# ============================================================================= 9. tokenization
hr("9. TOKENIZATION")
enc_train = [encode(s, l) for s, l in zip(train_df.event_sequence, train_df.label)]
enc_val = [encode(s, l) for s, l in zip(val_df.event_sequence, val_df.label)]
lens = np.array([len(e["input_ids"]) for e in enc_train + enc_val])
print(f"Token length (prompt+answer) min={lens.min()} median={int(np.median(lens))} "
      f"p95={int(np.percentile(lens, 95))} p99={int(np.percentile(lens, 99))} max={lens.max()}")
MAX_LEN = int(min(MAX_LEN_CAP, math.ceil(np.percentile(lens, 99) / 64) * 64))
print(f"Chosen max_length = {MAX_LEN} (p99 rounded up to 64, capped at {MAX_LEN_CAP}). Longer train/val examples "
      "are DROPPED (not truncated, which would corrupt the label/answer). Test examples are never dropped.")


def filt(enc, df, nm):
    keep = [i for i, e in enumerate(enc) if len(e["input_ids"]) <= MAX_LEN]
    dropped = df.iloc[[i for i in range(len(enc)) if i not in set(keep)]]
    print(f"  {nm}: kept {len(keep)}/{len(enc)}; dropped by class: {dropped.label.value_counts().to_dict()}")
    return [enc[i] for i in keep]


enc_train, enc_val = filt(enc_train, train_df, "train"), filt(enc_val, val_df, "val")


class ListDS(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]


train_ds, val_ds = ListDS(enc_train), ListDS(enc_val)
print("Dataset keys:", list(enc_train[0].keys()))
collator_tok = AutoTokenizer.from_pretrained(MODEL_NAME)
collator_tok.padding_side = "right"
if collator_tok.pad_token is None:
    collator_tok.pad_token = collator_tok.eos_token
collator = DataCollatorForSeq2Seq(collator_tok, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8)


# ============================================================================= model helpers / eval
def load_base():
    try:
        m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE)
    except TypeError:
        m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE)
    return m.to(DEVICE)


def parse_pred(text):
    t = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    if "<think>" in t:                      # unclosed reasoning block -> no final answer
        return None
    found = {m.lower() for m in re.findall(r"\b(normal|anomaly)\b", t, flags=re.I)}
    return {"normal": NORMAL, "anomaly": ANOMALY}[found.pop()] if len(found) == 1 else None


@torch.no_grad()
def generate_all(model, seqs, max_new=GEN_MAX_NEW_TOKENS, budget=16384, max_bs=64, tag=""):
    model.eval()
    prompts = [prompt_text(s) for s in seqs]
    plens = [len(tok(p, add_special_tokens=False).input_ids) for p in prompts]
    order = np.argsort(plens)[::-1]
    outs, i, nb, t0 = [None] * len(prompts), 0, 0, time.time()
    amp = torch.autocast("cuda", dtype=torch.float16) if USE_FP16_AMP else contextlib.nullcontext()
    while i < len(order):
        bs = int(max(1, min(max_bs, budget // max(plens[order[i]], 1))))
        idx = order[i:i + bs]
        batch = tok([prompts[j] for j in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        with amp:
            gen = model.generate(**batch, max_new_tokens=max_new, do_sample=False, temperature=None,
                                 top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
        for j, t in zip(idx, tok.batch_decode(gen[:, batch.input_ids.shape[1]:], skip_special_tokens=True)):
            outs[j] = t
        i += bs
        nb += 1
        if nb % 20 == 0:
            print(f"  [{tag}] {i}/{len(order)} generated ({time.time() - t0:.0f}s)")
    return outs


def metrics_for(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    labs = [NORMAL, ANOMALY]
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labs, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[NORMAL, ANOMALY, INVALID])[:2]
    return {
        "n": int(len(y_true)), "n_normal": int((y_true == NORMAL).sum()), "n_anomaly": int((y_true == ANOMALY).sum()),
        "n_invalid": int((y_pred == INVALID).sum()),
        "accuracy": float((y_true == y_pred).mean()),
        "precision": float(p[1]), "recall": float(r[1]), "f1": float(f[1]),      # Anomaly = positive class
        "macro_f1": float(f.mean()), "weighted_f1": float((f * s).sum() / max(s.sum(), 1)),
        "per_class": {l: {"precision": float(p[k]), "recall": float(r[k]), "f1": float(f[k]), "support": int(s[k])}
                      for k, l in enumerate(labs)},
        "confusion_matrix": {"rows_true": labs, "cols_pred": [NORMAL, ANOMALY, INVALID], "matrix": cm.tolist()},
    }


def score(model, df, tag):
    raw = generate_all(model, df.event_sequence.tolist(), tag=tag)
    pred = [parse_pred(t) or INVALID for t in raw]
    return raw, pred, metrics_for(df.label.values, pred)


# ============================================================================= 14. baselines (pre-LoRA)
hr("14. BASELINES (same test set, same prompt)")
y_test = test_df.label.values
maj = train_full.label.value_counts().idxmax()
m_major = metrics_for(y_test, [maj] * len(y_test))
print(f"Majority class (from full training split): {maj}")
print("Evaluating base Qwen3-0.6B (no fine-tuning)...")
base = load_base()
raw_base, pred_base, m_base = score(base, test_df, "base")
print("Sample base outputs:", [t[:40] for t in raw_base[:5]])
del base
gc.collect(); torch.cuda.empty_cache()

# ============================================================================= 7/8/10/11. LoRA train
hr("7-8. LORA CONFIG")
lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
                      task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])

if CUDA:
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    tok_budget = 8192 if mem >= 30 else (4096 if mem >= 14 else 2048)
    bs = int(min(32, 2 ** int(math.log2(max(1, tok_budget // MAX_LEN)))))
else:
    bs = 4
accum = math.ceil(EFFECTIVE_BS / bs)
use_gc = False
eval_key = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters \
    else "evaluation_strategy"

hr("10-11. TRAINING")
trainer = None
for attempt in range(6):
    shutil.rmtree(CKPT_DIR, ignore_errors=True)
    set_seed(SEED)
    base = load_base()
    if use_gc:
        base.config.use_cache = False
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        base.enable_input_require_grads()
    model = get_peft_model(base, lora_cfg)
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    if attempt == 0:
        print(f"Trainable params: {tr:,} | total params: {tot:,} | trainable: {100 * tr / tot:.4f}%")
    steps_per_epoch = math.ceil(len(train_ds) / (bs * accum))
    args = TrainingArguments(
        output_dir=CKPT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=bs,
        per_device_eval_batch_size=bs, gradient_accumulation_steps=accum, learning_rate=LR,
        weight_decay=WEIGHT_DECAY, warmup_steps=int(0.03 * steps_per_epoch * EPOCHS), lr_scheduler_type="cosine",
        logging_steps=10, save_strategy="epoch", save_total_limit=2, load_best_model_at_end=True,
        metric_for_best_model="eval_loss", greater_is_better=False, bf16=USE_BF16, fp16=USE_FP16_AMP,
        seed=SEED, data_seed=SEED, report_to="none", remove_unused_columns=False, dataloader_num_workers=0,      # <-- Set to 0 for Windows
    disable_tqdm=False,  
        **{eval_key: "epoch"})
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collator)
    print(f"Attempt {attempt + 1}: per-device batch={bs}, grad-accum={accum} (effective {bs * accum}), "
          f"epochs={EPOCHS}, lr={LR}, max_len={MAX_LEN}, grad-checkpointing={use_gc}, "
          f"precision={'bf16' if USE_BF16 else ('fp16-AMP' if USE_FP16_AMP else 'fp32')}")
    oom = False
    try:
        train_res = trainer.train()
        break
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if "out of memory" not in str(e).lower():
            raise
        oom = True
    if oom:
        del trainer, model, base
        gc.collect(); torch.cuda.empty_cache()
        if bs > 1:
            bs //= 2
            accum = math.ceil(EFFECTIVE_BS / bs)
            print(f"[CUDA OOM] retrying from scratch with per-device batch={bs}, grad-accum={accum}")
        elif not use_gc:
            use_gc = True
            print("[CUDA OOM at batch=1] retrying from scratch with gradient checkpointing")
        else:
            raise SystemExit("OOM even at batch=1 with gradient checkpointing; reduce MAX_LEN_CAP.")
else:
    raise SystemExit("Training failed after repeated OOM retries.")

hist = trainer.state.log_history
print("\nTraining log (step | epoch | loss | lr)")
for h in hist:
    if "loss" in h:
        print(f"  {h['step']:>5} | {h['epoch']:.2f} | {h['loss']:.4f} | {h.get('learning_rate', float('nan')):.2e}")
    if "eval_loss" in h:
        print(f"  >>> eval @ epoch {h['epoch']:.2f}: val_loss={h['eval_loss']:.4f}")
final_train_loss = next((h["loss"] for h in reversed(hist) if "loss" in h), float("nan"))
final_val_loss = next((h["eval_loss"] for h in reversed(hist) if "eval_loss" in h), float("nan"))
runtime = float(train_res.metrics.get("train_runtime", float("nan")))
print(f"Train runtime: {runtime:.0f}s | final train loss (last logged): {final_train_loss:.4f} | "
      f"final val loss: {final_val_loss:.4f} | mean train loss: {train_res.metrics.get('train_loss', float('nan')):.4f}")

hr("17. SAVE ADAPTER")
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
json.dump({"prompt": PROMPT, "base_model": MODEL_NAME, "enable_thinking": False, "labels": [NORMAL, ANOMALY],
           "max_len": MAX_LEN, "gen_max_new_tokens": GEN_MAX_NEW_TOKENS},
          open(os.path.join(ADAPTER_DIR, "experiment_config.json"), "w"), indent=2)
if templates:
    json.dump(templates, open(os.path.join(ADAPTER_DIR, "hdfs_templates.json"), "w"), indent=1)
print("Saved adapter + tokenizer to", ADAPTER_DIR, os.listdir(ADAPTER_DIR))

# ============================================================================= 12/13. evaluate fine-tuned
hr("12-13. EVALUATION OF FINE-TUNED MODEL (held-out test)")
model.config.use_cache = True
raw_ft, pred_ft, m_ft = score(model, test_df, "finetuned")

pred_df = pd.DataFrame({"block_id": test_df.block_id, "event_sequence": test_df.event_sequence,
                        "true_label": test_df.label, "predicted_label": pred_ft,
                        "valid_prediction": [p != INVALID for p in pred_ft], "raw_output": raw_ft})
pred_df.to_csv(PRED_CSV, index=False)

# ---- confusion matrix figure
cm = np.array(m_ft["confusion_matrix"]["matrix"])
cols = [NORMAL, ANOMALY] + ([INVALID] if m_ft["n_invalid"] > 0 else [])
cm_plot = cm[:, :len(cols)]
fig, ax = plt.subplots(figsize=(5.2, 4.2))
ax.imshow(cm_plot, cmap="Blues")
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols)
ax.set_yticks(range(2)); ax.set_yticklabels([NORMAL, ANOMALY])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("HDFS Qwen3-0.6B LoRA - test set")
for i in range(cm_plot.shape[0]):
    for j in range(cm_plot.shape[1]):
        ax.text(j, i, f"{cm_plot[i, j]:,}", ha="center", va="center",
                color="white" if cm_plot[i, j] > cm_plot.max() / 2 else "black")
plt.tight_layout(); plt.savefig(CM_PNG, dpi=200); plt.close()

# ---- print metrics
def show(nm, m):
    print(f"\n{nm}: n={m['n']} (Normal {m['n_normal']}, Anomaly {m['n_anomaly']}) invalid={m['n_invalid']}")
    print(f"  Accuracy={m['accuracy']:.4f}  Precision(Anomaly)={m['precision']:.4f}  Recall(Anomaly)={m['recall']:.4f}  "
          f"F1(Anomaly)={m['f1']:.4f}  MacroF1={m['macro_f1']:.4f}  WeightedF1={m['weighted_f1']:.4f}")
    for c, v in m["per_class"].items():
        print(f"  {c:<8} P={v['precision']:.4f} R={v['recall']:.4f} F1={v['f1']:.4f} support={v['support']}")
    print("  confusion (rows=actual Normal/Anomaly; cols=pred Normal/Anomaly/Invalid):", m["confusion_matrix"]["matrix"])


show("Majority baseline", m_major); show("Base Qwen3-0.6B", m_base); show("Fine-tuned Qwen3-0.6B", m_ft)
print("\n" + pd.DataFrame({k: {c: round(m[c], 4) for c in ["accuracy", "precision", "recall", "f1", "macro_f1", "weighted_f1"]}
                            for k, m in [("Majority", m_major), ("Base Qwen3-0.6B", m_base),
                                         ("Fine-tuned Qwen3-0.6B", m_ft)]}).T.to_string())
print("(precision/recall/F1 = Anomaly class as positive; invalid outputs count as wrong)")

# seen vs unseen sequences
unseen_mask = ~test_df.event_sequence.isin(train_seq_set).values
extra = {}
if unseen_mask.sum() > 0:
    sub = test_df[unseen_mask]
    extra = {"n": int(unseen_mask.sum()),
             "majority": metrics_for(sub.label.values, [maj] * len(sub)),
             "base": metrics_for(sub.label.values, np.array(pred_base)[unseen_mask]),
             "finetuned": metrics_for(sub.label.values, np.array(pred_ft)[unseen_mask])}
    print(f"\nTest traces whose sequence never occurs in training: {extra['n']} "
          f"(Anomaly {extra['finetuned']['n_anomaly']}); fine-tuned acc={extra['finetuned']['accuracy']:.4f} "
          f"anomaly-F1={extra['finetuned']['f1']:.4f}")

# ============================================================================= 16. error analysis
hr("16. ERROR ANALYSIS")
ev = pred_df.assign(n_events=test_df.n_events.values, correct=(pred_df.true_label == pred_df.predicted_label).values)
errs = ev[~ev.correct]
print(f"Errors: {len(errs)} / {len(ev)}  | false negatives (Anomaly->Normal/Invalid): "
      f"{((errs.true_label == ANOMALY)).sum()} | false positives (Normal->Anomaly/Invalid): "
      f"{((errs.true_label == NORMAL)).sum()} | invalid: {(~errs.valid_prediction).sum()}")
fn = errs[errs.true_label == ANOMALY].head(6)
fp = errs[errs.true_label == NORMAL].head(6)
shown = pd.concat([fn, fp])
if len(shown) < 10:
    shown = pd.concat([shown, errs.drop(shown.index).head(10 - len(shown))])
for _, r in shown.iterrows():
    e = r.event_sequence.split()
    print(f"Block: {r.block_id} | events={len(e)}\n  Sequence: {' '.join(e[:30])}{' ...(+%d)' % (len(e) - 30) if len(e) > 30 else ''}"
          f"\n  Truth: {r.true_label} | Prediction: {r.predicted_label} | raw={r.raw_output[:30]!r}")

print("\nObserved statistics (no causal claims):")
print(f"  median #events  correct={ev[ev.correct].n_events.median():.0f}  errors={errs.n_events.median() if len(errs) else float('nan'):.0f}")
bins = [0, 10, 20, 40, 80, 10**9]
ev["bucket"] = pd.cut(ev.n_events, bins, labels=["<=10", "11-20", "21-40", "41-80", ">80"])
print(ev.groupby("bucket", observed=True).agg(n=("correct", "size"), error_rate=("correct", lambda s: 1 - s.mean()),
                                              anomaly_share=("true_label", lambda s: (s == ANOMALY).mean())).round(4).to_string())
train_labels_by_seq = train_full.groupby("event_sequence").label.agg(lambda s: set(s)).to_dict()
ev["seen_in_train"] = ev.event_sequence.isin(train_labels_by_seq)
ev["ambiguous_in_train"] = ev.event_sequence.map(lambda s: len(train_labels_by_seq.get(s, ())) > 1)
train_bigrams = set()
for s in train_seq_set:
    w = s.split()
    train_bigrams.update(zip(w, w[1:]))
ev["unseen_bigrams"] = ev.event_sequence.map(lambda s: sum(1 for b in zip(s.split(), s.split()[1:]) if b not in train_bigrams))
for nm, d in [("all test", ev), ("errors", ev[~ev.correct])]:
    if len(d):
        print(f"  {nm}: seen-in-train={d.seen_in_train.mean():.1%} | ambiguous(both labels in train)={d.ambiguous_in_train.mean():.1%} "
              f"| has unseen event bigram={(d.unseen_bigrams > 0).mean():.1%}")

# ============================================================================= 17. metrics json
def clean(o):
    if isinstance(o, dict): return {k: clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return o


n_all = len(traces)
nn_all, na_all = split_counts(traces)
report = {
    "dataset": {"n_traces": n_all, "normal": nn_all, "anomaly": na_all, "label_source": label_src,
                "split_full": {k: dict(zip(["normal", "anomaly"], split_counts(d)), total=len(d))
                               for k, d in [("train", train_full), ("val", val_full), ("test", test_full)]},
                "split_used": {k: dict(zip(["normal", "anomaly"], split_counts(d)), total=len(d))
                               for k, d in [("train", train_df), ("val", val_df), ("test", test_df)]},
                "train_anomaly_frac": TRAIN_ANOMALY_FRAC, "test_seq_seen_in_train_frac": float(overlap)},
    "model": MODEL_NAME,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "targets": lora_cfg.target_modules if not isinstance(lora_cfg.target_modules, set) else sorted(lora_cfg.target_modules)},
    "training": {"epochs": EPOCHS, "lr": LR, "weight_decay": WEIGHT_DECAY, "per_device_batch": bs, "grad_accum": accum,
                 "max_len": MAX_LEN, "runtime_s": runtime, "final_train_loss": final_train_loss,
                 "final_val_loss": final_val_loss, "seed": SEED},
    "results": {"majority_baseline": m_major, "base_qwen": m_base, "finetuned_qwen": m_ft},
    "unseen_sequence_subset": extra,
}
json.dump(clean(report), open(METRICS_JSON, "w"), indent=2)

# ============================================================================= 18. reload test
hr("18. RELOAD TEST")
del trainer, model, base
gc.collect(); torch.cuda.empty_cache()
tok_rl = AutoTokenizer.from_pretrained(ADAPTER_DIR)
assert tok_rl.pad_token is not None
tok_rl.padding_side = "left"
base_rl = load_base()
model_rl = PeftModel.from_pretrained(base_rl, ADAPTER_DIR).eval()
n_chk = min(200, len(test_df))
raw_rl = generate_all(model_rl, test_df.event_sequence.tolist()[:n_chk], tag="reload")
pred_rl = [parse_pred(t) or INVALID for t in raw_rl]
agree = np.mean([a == b for a, b in zip(pred_rl, pred_ft[:n_chk])])
acc_rl = np.mean([a == b for a, b in zip(pred_rl, test_df.label.values[:n_chk])])
print(f"Reloaded adapter produced {sum(p != INVALID for p in pred_rl)}/{n_chk} valid predictions | "
      f"agreement with in-memory fine-tuned model: {agree:.1%} | accuracy on these {n_chk}: {acc_rl:.1%}")

reload_code = f'''import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
tok = AutoTokenizer.from_pretrained("{ADAPTER_DIR}"); tok.padding_side = "left"
base = AutoModelForCausalLM.from_pretrained("{MODEL_NAME}", dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32).to("cuda")
model = PeftModel.from_pretrained(base, "{ADAPTER_DIR}").eval()
PROMPT = {PROMPT!r}
def classify(events):   # events: "E5 E22 E5 E11 ..."
    msgs = [{{"role": "user", "content": PROMPT.format(seq=events)}}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    x = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**x, max_new_tokens=16, do_sample=False)
    return tok.decode(out[0, x.input_ids.shape[1]:], skip_special_tokens=True).strip()
print(classify("E5 E22 E5 E11 E9 E11 E9 E26 E26 E26"))
'''
open(f"{ROOT}/reload_adapter.py", "w").write(reload_code)

# ============================================================================= 19. summary
hr("HDFS QWEN3-0.6B LORA EXPERIMENT")
f = lambda m, k: f"{m[k]:.4f}"
print(f"""Dataset: HDFS_v1 (labels: {label_src})
  Number of traces: {n_all:,}
  Normal: {nn_all:,}
  Anomaly: {na_all:,}
Split (block-level, stratified={strat_ok}, seed={SEED}): full 80/10/10, subsampled for compute
  Train: {len(train_df):,} of {len(train_full):,} (anomaly frac {TRAIN_ANOMALY_FRAC})
  Validation: {len(val_df):,} of {len(val_full):,}
  Test: {len(test_df):,} of {len(test_full):,} (natural class ratio)
Model: {MODEL_NAME}
Fine-tuning: LoRA
  Rank: {LORA_R} | Alpha: {LORA_ALPHA} | Dropout: {LORA_DROPOUT}
Training:
  Epochs: {EPOCHS} | Learning rate: {LR} | Runtime: {runtime:.0f}s
  Final training loss: {final_train_loss:.4f} | Final validation loss: {final_val_loss:.4f}

RESULTS (test; Precision/Recall/F1 = Anomaly class)
----------------------------------------
                       Accuracy  Precision  Recall  F1      MacroF1  WeightedF1  Invalid
Majority ({maj:<7})      {f(m_major,'accuracy')}    {f(m_major,'precision')}     {f(m_major,'recall')}  {f(m_major,'f1')}  {f(m_major,'macro_f1')}   {f(m_major,'weighted_f1')}      {m_major['n_invalid']}
Base Qwen3-0.6B        {f(m_base,'accuracy')}    {f(m_base,'precision')}     {f(m_base,'recall')}  {f(m_base,'f1')}  {f(m_base,'macro_f1')}   {f(m_base,'weighted_f1')}      {m_base['n_invalid']}
Fine-tuned Qwen3-0.6B  {f(m_ft,'accuracy')}    {f(m_ft,'precision')}     {f(m_ft,'recall')}  {f(m_ft,'f1')}  {f(m_ft,'macro_f1')}   {f(m_ft,'weighted_f1')}      {m_ft['n_invalid']}

Artifacts:
  LoRA adapter: {ADAPTER_DIR}
  Predictions: {PRED_CSV}
  Metrics: {METRICS_JSON}
  Confusion matrix: {CM_PNG}
  Reload script: {ROOT}/reload_adapter.py
""")
print("Improvement over base (F1 / macro F1):",
      f"{m_ft['f1'] - m_base['f1']:+.4f} / {m_ft['macro_f1'] - m_base['macro_f1']:+.4f}",
      "| over majority macro F1:", f"{m_ft['macro_f1'] - m_major['macro_f1']:+.4f}")
print("\nTo reload the adapter:\n" + reload_code)
print("=" * 78)

c:\Users\nikhi\OneDrive\Attachments\Desktop\Git_Repos\Ml_Hackathon\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True | NVIDIA GeForce RTX 4060 Laptop GPU

1. DATASET DISCOVERY
Discovered 5 HDFS-related files

c:\Users\nikhi\Downloads\hdfs_data\HDFS_v1.zip  [178.0MB]

c:\Users\nikhi\Downloads\hdfs_data\preprocessed\Event_traces.csv  [119.6MB]
  rows: 575,061   columns: ['BlockId', 'Label', 'Type', 'Features', 'TimeInterval', 'Latency']
                      BlockId    Label  Type                                                               Features                                                           TimeInterval  Latency
  0  blk_-1608999687919862906  Success   NaN  [E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26,E6,E5,E16,E6,E5,E18...  [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, ...     3802
  1   blk_7503483334202473044  Success   NaN  [E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26,E3,E2,E2,E23,E23,E2...  [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, ...     3802
  2  blk_-3544583377289625738     Fail  21.0  [E5,E22,E5,E5,E11,E9,E1

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 556.53it/s]


  [base] 1241/6000 generated (17s)
  [base] 2521/6000 generated (33s)
  [base] 3801/6000 generated (49s)
  [base] 5081/6000 generated (64s)
Sample base outputs: ['Anomaly', 'Normal', 'Anomaly', 'Normal', 'Anomaly']

7-8. LORA CONFIG

10-11. TRAINING


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 573.83it/s]


Trainable params: 4,587,520 | total params: 600,637,440 | trainable: 0.7638%
Attempt 1: per-device batch=8, grad-accum=4 (effective 32), epochs=2, lr=0.0002, max_len=192, grad-checkpointing=False, precision=bf16


Epoch,Training Loss,Validation Loss
1,0.014221,0.004119
2,0.007838,0.003062



Training log (step | epoch | loss | lr)
     10 | 0.04 | 0.2856 | 1.20e-04
     20 | 0.08 | 0.1906 | 2.00e-04
     30 | 0.12 | 0.0895 | 2.00e-04
     40 | 0.16 | 0.0672 | 1.99e-04
     50 | 0.20 | 0.3074 | 1.98e-04
     60 | 0.24 | 0.0735 | 1.96e-04
     70 | 0.28 | 0.0249 | 1.94e-04
     80 | 0.32 | 0.0451 | 1.92e-04
     90 | 0.36 | 0.0479 | 1.89e-04
    100 | 0.40 | 0.0263 | 1.86e-04
    110 | 0.44 | 0.1562 | 1.82e-04
    120 | 0.48 | 0.1319 | 1.78e-04
    130 | 0.52 | 0.0298 | 1.74e-04
    140 | 0.56 | 0.0037 | 1.69e-04
    150 | 0.60 | 0.0028 | 1.65e-04
    160 | 0.64 | 0.0171 | 1.60e-04
    170 | 0.68 | 0.0359 | 1.54e-04
    180 | 0.72 | 0.0118 | 1.49e-04
    190 | 0.76 | 0.0135 | 1.43e-04
    200 | 0.80 | 0.0101 | 1.37e-04
    210 | 0.84 | 0.0170 | 1.31e-04
    220 | 0.88 | 0.0088 | 1.25e-04
    230 | 0.92 | 0.0236 | 1.18e-04
    240 | 0.96 | 0.0080 | 1.12e-04
    250 | 1.00 | 0.0142 | 1.06e-04
  >>> eval @ epoch 1.00: val_loss=0.0041
    260 | 1.04 | 0.0015 | 9.90e-05
    270 

Loading weights: 100%|██████████| 311/311 [00:01<00:00, 270.23it/s]


Reloaded adapter produced 200/200 valid predictions | agreement with in-memory fine-tuned model: 100.0% | accuracy on these 200: 99.5%

HDFS QWEN3-0.6B LORA EXPERIMENT
Dataset: HDFS_v1 (labels: anomaly_label.csv)
  Number of traces: 575,061
  Normal: 558,223
  Anomaly: 16,838
Split (block-level, stratified=True, seed=42): full 80/10/10, subsampled for compute
  Train: 8,000 of 460,048 (anomaly frac 0.25)
  Validation: 600 of 57,506
  Test: 6,000 of 57,507 (natural class ratio)
Model: Qwen/Qwen3-0.6B
Fine-tuning: LoRA
  Rank: 16 | Alpha: 32 | Dropout: 0.05
Training:
  Epochs: 2 | Learning rate: 0.0002 | Runtime: 1029s
  Final training loss: 0.0078 | Final validation loss: 0.0031

RESULTS (test; Precision/Recall/F1 = Anomaly class)
----------------------------------------
                       Accuracy  Precision  Recall  F1      MacroF1  WeightedF1  Invalid
Majority (Normal )      0.9707    0.0000     0.0000  0.0000  0.4926   0.9562      0
Base Qwen3-0.6B        0.7240    0.0443     0.

In [ ]:
!pip install -q -U "torchao>=0.16.0" peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 75.3 MB/s eta 0:00:00


In [ ]:
import torch
import torchao
import peft
import transformers

print("torch:", torch.__version__)
print("torchao:", torchao.__version__)
print("peft:", peft.__version__)
print("transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

torch: 2.11.0+cu128
torchao: 0.18.0
peft: 0.21.2
transformers: 5.18.0
CUDA: True
